# Week 43–44: frozen QA challenge inference

This notebook recovers the preserved **1 October** file revisions into a separate Colab folder, verifies the original three-epoch decision rule, then evaluates that recovered model. It does not train or tune anything. The current Drive files are not replaced.

1. Select a GPU runtime and run all cells.
2. Approve Google sign-in for Drive mounting and version downloads.
3. Upload the updated `week43-challenge-runner-recovery.zip` when prompted.
4. Wait for checkpoint recovery (about 1.1 GB) and inference.
5. Send the downloaded `week43_qa_challenge_results.zip` back to the project chat.

Challenge results use seed 20260901 and its original saved cutoff, not the three-seed validation mean. Language-specific thresholds remain frozen before challenge inference. Downloads use Google's documented Drive revision API: https://developers.google.com/workspace/drive/api/guides/manage-downloads


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')


In [ ]:
from google.colab import files
import zipfile, os, subprocess, sys
uploaded = files.upload()  # Choose week43-challenge-runner-recovery.zip
archive = next(iter(uploaded))
WORK = Path('/content/nlp-final-challenge')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(WORK)
os.chdir(WORK)
print(Path.cwd())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.17.0', 'scikit-learn==1.7.2', 'tabulate', 'google-api-python-client'], check=True)


In [ ]:
from scripts.recover_week40_checkpoint import recover_checkpoint
CHECKPOINT = str(recover_checkpoint())


In [ ]:
def run_with_output(command):
    completed = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"Evaluation stopped (exit {completed.returncode}). "
            "Copy the complete output above, including the first traceback, "
            "back to the project chat."
        )

run_with_output([sys.executable, 'student_starter/validate_challenge.py', 'challenge.json'])
run_with_output([sys.executable, '-u', '-m', 'scripts.evaluate_challenge',
                 '--qa-only', '--qa-checkpoint', CHECKPOINT])


In [ ]:
import shutil
result_zip = shutil.make_archive('/content/week43_qa_challenge_results', 'zip', WORK, 'results/challenge')
files.download(result_zip)
